# Cantonese ELECTRA-large: full fine-tuning control vs LoRA

This Run-all notebook performs one controlled **full fine-tuning** run from the original Stanza 1.14 Mandarin ELECTRA dependency checkpoint. It updates the complete Transformer and parsing layers. POS and lemma processors remain frozen, and the exact custom train/dev split, predicted POS/lemma inputs, learning rates, maximum steps, evaluation interval, patience, seed, and dev CoNLL-2018 LAS selection rule match the prior LoRA run.

The notebook does **not** evaluate test. It uploads the existing `yue_dev_diagnostics.zip` only to obtain the verified LoRA dev prediction, compares LoRA and full fine-tuning token by token, and exports a deterministic 36-sentence manual review sample covering distance ranges, principal low-scoring relations, fixed errors, regressions, and persistent errors.

Use an A100 GPU if available. Full ELECTRA-large fine-tuning and its optimizer require much more memory and storage than LoRA. A single matched run answers whether full fine-tuning wins under this schedule; it cannot establish a universal LoRA capacity limitation.

In [ ]:
# Pinned Python packages.  Restart the runtime after this cell if Colab asks.
%pip install -q stanza==1.14.0 transformers==4.56.2 huggingface-hub==0.34.4


In [ ]:
from pathlib import Path
import os, json, hashlib, random, shutil, gc, copy, importlib.util, logging
import numpy as np
import torch

SEED = 42
WORK = Path('/content/yue_full_finetune_comparison')
DATA = WORK / 'data'
MODELS = WORK / 'stanza_resources_1.14.0'
OUT = WORK / 'outputs'
HF_HOME = WORK / 'hf_home'
for p in (DATA, MODELS, OUT, HF_HOME): p.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(HF_HOME)

RAW_URL = 'https://raw.githubusercontent.com/UniversalDependencies/UD_Cantonese-HK/r2.18/yue_hk-ud-test.conllu'
RAW_SHA256 = 'cbd843a195d0db4cdafbf6fcafb7b7b559afea750411006f4728311e70cc4e2a'
SPLIT_POSITIONS = {'train': [1, 2, 3, 5, 6, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 27, 29, 30, 32, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 48, 49, 51, 52, 53, 54, 55, 56, 57, 58, 60, 61, 62, 63, 64, 65, 66, 68, 70, 71, 72, 74, 76, 77, 78, 79, 80, 81, 84, 85, 86, 87, 88, 89, 90, 92, 93, 94, 98, 99, 100, 102, 103, 104, 107, 108, 109, 110, 111, 112, 113, 115, 116, 118, 119, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 150, 151, 152, 154, 155, 156, 157, 158, 159, 160, 161, 162, 164, 165, 167, 169, 171, 172, 175, 176, 177, 178, 179, 180, 181, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 205, 206, 207, 208, 209, 210, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 223, 226, 227, 229, 230, 234, 235, 237, 238, 239, 240, 243, 244, 246, 248, 249, 251, 252, 253, 254, 255, 256, 257, 258, 259, 261, 263, 264, 265, 266, 267, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 284, 287, 289, 290, 291, 292, 295, 296, 298, 299, 300, 301, 302, 303, 304, 305, 306, 307, 308, 310, 311, 312, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 339, 340, 341, 342, 343, 344, 345, 346, 347, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 362, 363, 364, 365, 366, 367, 368, 370, 371, 372, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 386, 387, 388, 390, 391, 393, 394, 396, 397, 398, 399, 400, 402, 403, 404, 405, 406, 407, 408, 413, 414, 415, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 428, 429, 431, 433, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 454, 455, 456, 458, 459, 460, 461, 462, 464, 465, 466, 467, 468, 469, 471, 472, 473, 474, 475, 476, 477, 478, 479, 480, 481, 482, 483, 485, 486, 487, 488, 490, 491, 492, 493, 496, 497, 499, 500, 501, 502, 503, 505, 506, 507, 508, 509, 510, 511, 512, 513, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 524, 525, 526, 527, 528, 529, 531, 532, 533, 534, 535, 537, 538, 539, 540, 541, 543, 544, 545, 547, 548, 549, 550, 551, 552, 553, 554, 555, 556, 557, 558, 559, 560, 561, 562, 563, 564, 565, 566, 568, 569, 570, 571, 572, 573, 575, 576, 578, 580, 581, 582, 583, 584, 585, 587, 588, 589, 590, 591, 592, 593, 596, 597, 598, 600, 601, 602, 603, 605, 606, 607, 608, 609, 611, 612, 613, 614, 615, 616, 617, 618, 619, 620, 622, 623, 624, 625, 626, 629, 630, 631, 632, 635, 636, 637, 638, 639, 641, 642, 643, 644, 645, 647, 649, 650, 651, 652, 654, 655, 656, 657, 658, 659, 660, 661, 662, 664, 665, 666, 667, 668, 669, 670, 671, 674, 675, 676, 677, 678, 679, 681, 682, 683, 687, 689, 690, 691, 692, 694, 696, 697, 698, 699, 700, 702, 703, 704, 705, 706, 708, 709, 710, 711, 712, 713, 714, 715, 717, 718, 719, 720, 721, 723, 724, 725, 727, 730, 732, 733, 735, 736, 737, 738, 739, 740, 741, 742, 743, 745, 746, 747, 749, 750, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 762, 764, 767, 770, 771, 772, 773, 774, 775, 778, 779, 780, 781, 782, 783, 785, 786, 787, 790, 791, 793, 795, 796, 797, 799, 800, 801, 802, 804, 805, 806, 808, 809, 810, 812, 813, 814, 815, 818, 819, 820, 822, 823, 825, 826, 827, 828, 829, 830, 831, 832, 833, 834, 836, 838, 839, 840, 841, 842, 843, 844, 845, 846, 847, 848, 849, 850, 851, 852, 853, 857, 859, 860, 861, 862, 863, 864, 865, 866, 867, 868, 869, 870, 872, 873, 874, 875, 876, 877, 878, 880, 881, 882, 883, 884, 886, 888, 889, 890, 891, 892, 893, 894, 895, 896, 897, 899, 900, 901, 902, 903, 904, 907, 908, 909, 910, 911, 912, 915, 916, 917, 919, 920, 922, 923, 924, 925, 926, 927, 928, 929, 930, 933, 934, 935, 937, 938, 939, 940, 942, 943, 945, 946, 947, 948, 949, 950, 951, 953, 954, 955, 956, 957, 958, 962, 964, 965, 966, 967, 968, 970, 971, 972, 973, 974, 976, 977, 978, 980, 981, 982, 984, 985, 986, 989, 990, 992, 993, 994, 996, 998, 1000, 1001, 1002, 1004], 'dev': [4, 12, 50, 59, 67, 69, 75, 95, 114, 120, 148, 153, 163, 168, 174, 182, 222, 224, 225, 232, 242, 262, 268, 281, 283, 285, 286, 288, 337, 338, 348, 410, 412, 416, 427, 430, 432, 434, 453, 463, 470, 494, 498, 504, 530, 536, 546, 567, 574, 579, 595, 599, 610, 627, 628, 640, 653, 672, 685, 686, 688, 693, 701, 716, 726, 728, 729, 731, 734, 765, 766, 768, 769, 776, 792, 794, 798, 803, 816, 817, 824, 835, 837, 854, 871, 887, 905, 914, 931, 932, 936, 941, 960, 961, 963, 975, 979, 987, 997, 999, 1003], 'test': [7, 26, 28, 31, 33, 45, 47, 73, 82, 83, 91, 96, 97, 101, 105, 106, 117, 132, 149, 166, 170, 173, 204, 211, 228, 231, 233, 236, 241, 245, 247, 250, 260, 282, 293, 294, 297, 309, 313, 361, 369, 373, 385, 389, 392, 395, 401, 409, 411, 452, 457, 484, 489, 495, 542, 577, 586, 594, 604, 621, 633, 634, 646, 648, 663, 673, 680, 684, 695, 707, 722, 744, 748, 751, 763, 777, 784, 788, 789, 807, 811, 821, 855, 856, 858, 879, 885, 898, 906, 913, 918, 921, 944, 952, 959, 969, 983, 988, 991, 995]}
SPLIT_SHA256 = {'train': 'b2d6b96af234f22825bb007a9a2a57ef49619c59747dc582dca7fdb1a4331a2d', 'dev': '41bc28d903457e70a4747307e56b3eb7820f2e6d3ede10ab549fd4a85aef63b7', 'test': '1d7b19ac4c0a75d58a80482413ff9cac63ed18917262c18fa1f92fc0f871c0a0'}
EVAL_URL = 'https://universaldependencies.org/conll18/conll18_ud_eval.py'
EVAL_SHA256 = '1072e02af00b1a56205b5e8216d51dee9b8944a104d80744afaccc78859fcb16'
HF_REPO = 'hfl/chinese-electra-180g-large-discriminator'
# If the earlier 41.32% run recorded a HF commit, paste it here.  None resolves HEAD once,
# records the exact commit, and then forces offline reuse for the rest of this run.
HF_ELECTRA_REVISION = 'd017e219578df8e4885484edbc8969dbdea9cbe0'

CFG = dict(parser_lr=1e-3, transformer_lr=2e-5, batch_size=900,
           max_steps=4000, eval_interval=100, patience_steps=600,
           max_grad_norm=1.0, selected_sentence_count=36)

def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1<<20), b''): h.update(b)
    return h.hexdigest()

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'ELECTRA-large training requires a Colab GPU runtime.'
print('device:', DEVICE, 'config:', CFG)


In [ ]:
# Download the pinned UD source and reconstruct the project's exact custom splits.
import urllib.request
raw_path = DATA / 'yue_hk-ud-test.r2.18.conllu'
urllib.request.urlretrieve(RAW_URL, raw_path)
assert sha256(raw_path) == RAW_SHA256

raw = raw_path.read_text(encoding='utf-8')
blocks = raw.strip().split('\n\n')
assert len(blocks) == 1004
for split in ('train','dev'):
    positions = SPLIT_POSITIONS[split]
    # CoNLL-U requires a blank line between sentence blocks.  The project files
    # also end in one blank line, hence the final two newline characters.
    text = '\n\n'.join(blocks[i-1] for i in positions) + '\n\n'
    path = DATA / f'{split}.conllu'
    path.write_text(text, encoding='utf-8')
    assert sha256(path) == SPLIT_SHA256[split], (split, sha256(path))
    print(split, len(positions), sha256(path))

eval_path = WORK / 'conll18_ud_eval.py'
urllib.request.urlretrieve(EVAL_URL, eval_path)
assert sha256(eval_path) == EVAL_SHA256
spec=importlib.util.spec_from_file_location('official_conll18', eval_path)
official=importlib.util.module_from_spec(spec); spec.loader.exec_module(official)

def conll18(path_gold, path_system):
    return official.evaluate(official.load_conllu_file(str(path_gold)),
                             official.load_conllu_file(str(path_system)))


In [ ]:
# Resolve one exact HF commit and download it into an isolated cache.
from huggingface_hub import snapshot_download
snapshot = Path(snapshot_download(HF_REPO, revision=HF_ELECTRA_REVISION, cache_dir=HF_HOME/'hub'))
HF_COMMIT = snapshot.name
assert len(HF_COMMIT) == 40
print('frozen HF revision:', HF_COMMIT)


In [ ]:
# Download the exact Stanza 1.14 Mandarin processors and verify registry/artifact hashes.
import stanza
from stanza.resources.common import download_resources_json, load_resources_json
from stanza.pipeline.core import DownloadMethod
assert stanza.__version__ == '1.14.0', f'Expected Stanza 1.14.0, found {stanza.__version__}. Restart runtime and rerun.'
download_resources_json(model_dir=str(MODELS))
resources_path=MODELS/'resources.json'
assert sha256(resources_path) == '4e41c1df152146fa26ed0c006a08feea7a60bb3414bb6d57dbda24ad2e3cb99c'
packages={'tokenize':'gsdsimp','pos':'gsdsimp_electra-large','lemma':'gsdsimp_charlm','depparse':'gsdsimp_electra-large'}
stanza.download('zh-hans', model_dir=str(MODELS), package=None, processors=packages, verbose=True)
resources=load_resources_json(model_dir=str(MODELS))
expected_md5={'tokenize':'48f993223d568afedc2893f7cd76719c','pos':'73859e5ec15bedc545d6deafd6ddba94','lemma':'b49edd41abb063a87b125ec53aa5b96c','depparse':'c7ea98d93459b22720337a9dcba1a848'}
artifacts={}
for proc,pkg in packages.items():
    p=MODELS/'zh-hans'/proc/f'{pkg}.pt'
    registry=resources['zh-hans'][proc][pkg]['md5']
    assert registry == expected_md5[proc] and p.exists()
    artifacts[proc]={'path':str(p),'registry_md5':registry,'sha256':sha256(p)}
print(json.dumps(artifacts, indent=2))
# Processor artifacts and the exact Transformer snapshot are now present.  Forbid
# later Hub lookups so every POS/parser load reuses this one cached revision.
os.environ['HF_HUB_OFFLINE']='1'
os.environ['TRANSFORMERS_OFFLINE']='1'


In [ ]:
# Run frozen POS/lemma exactly once and build pretagged train/dev caches.
# Gold sentence boundaries and integer-ID FORM tokenization are fed to the pipeline.
# Only LEMMA/UPOS/XPOS/FEATS are replaced; gold HEAD/DEPREL remain training targets.
tagger = stanza.Pipeline(lang='zh-hans', dir=str(MODELS),
    processors={k:v for k,v in packages.items() if k != 'depparse'},
    tokenize_pretokenized=True, use_gpu=True,
    download_method=DownloadMethod.REUSE_RESOURCES, verbose=False)
for proc in ('pos','lemma'):
    model=tagger.processors[proc]._trainer.model
    model.eval()
    for p in model.parameters(): p.requires_grad=False
    assert not any(p.requires_grad for p in model.parameters()), f'{proc} unexpectedly trainable'

def split_blocks(text): return text.strip().split('\n\n')
def integer_rows(block):
    return [line.split('\t') for line in block.splitlines()
            if line and not line.startswith('#') and line.split('\t',1)[0].isdigit()]

def make_pretagged(src, dst, chunk=32):
    bs=split_blocks(src.read_text(encoding='utf-8')); out=[]
    for start in range(0,len(bs),chunk):
        sub=bs[start:start+chunk]
        forms=[[r[1] for r in integer_rows(b)] for b in sub]
        doc=tagger(forms)
        assert len(doc.sentences)==len(sub)
        for block,sent,gold_forms in zip(sub,doc.sentences,forms):
            assert [w.text for w in sent.words] == gold_forms
            pred=iter(sent.words); lines=[]
            for line in block.splitlines():
                if line and not line.startswith('#') and line.split('\t',1)[0].isdigit():
                    c=line.split('\t'); w=next(pred)
                    c[2]=w.lemma or '_'; c[3]=w.upos or '_'; c[4]=w.xpos or '_'; c[5]=w.feats or '_'
                    line='\t'.join(c)
                lines.append(line)
            try: next(pred); raise AssertionError('extra predicted word')
            except StopIteration: pass
            out.append('\n'.join(lines))
    dst.write_text('\n\n'.join(out)+'\n',encoding='utf-8')
    assert len(split_blocks(dst.read_text()))==len(bs)

cache_hashes={}
for split in ('train','dev'):
    dst=DATA/f'{split}.predposlemma.conllu'
    make_pretagged(DATA/f'{split}.conllu',dst)
    cache_hashes[split]=sha256(dst)
    print(split,'frozen cache',cache_hashes[split])
del tagger; gc.collect(); torch.cuda.empty_cache()

EXPECTED_CACHE_SHA256={'train':'806b3d42c4f6838d6ba9c6565f29062a0d7b6e136e12d158e4449c3e7e970e89',
                       'dev':'e88aa03ff7453469deda68ed30c29dd16769e9839d2010b36abf91faace3bd97'}
assert cache_hashes == EXPECTED_CACHE_SHA256,(cache_hashes,EXPECTED_CACHE_SHA256)
print('Frozen POS/lemma caches exactly match the LoRA run: OK')


In [ ]:
# Construct the full-fine-tuning initialization from the original Mandarin checkpoint.
# Only new Cantonese DEPREL output units are initialized; every old parser tensor is preserved.
from stanza.models.pos.vocab import MultiVocab
from stanza.models.common.vocab import VOCAB_PREFIX_SIZE

base_path=Path(artifacts['depparse']['path'])
assert sha256(base_path)=='6531bcc2dbfbe1e3b19d4deb855533fc2379f7305162423f7b53216b1e03434'
base_ckpt=torch.load(base_path,map_location='cpu',weights_only=True)
assert base_ckpt.get('model_type','graph') == 'graph'
old_vocab=MultiVocab.load_state_dict(base_ckpt['vocab'])
old_units=list(old_vocab['deprel']._id2unit)
train_labels=sorted({r[7] for b in split_blocks((DATA/'train.conllu').read_text()) for r in integer_rows(b)})
new_labels=[x for x in train_labels if x not in old_vocab['deprel']]

expanded=copy.deepcopy(base_ckpt)
dep_state=expanded['vocab']['deprel']
dep_state['_id2unit']=old_units+new_labels
dep_state['_unit2id']={u:i for i,u in enumerate(dep_state['_id2unit'])}
old_n=len(old_units)-VOCAB_PREFIX_SIZE; new_n=old_n+len(new_labels)
expanded_names=[]
if new_labels:
    for name,t in list(expanded['model'].items()):
        if name == 'deprel.scorer.W_bilin.weight':
            z=t.new_zeros(t.shape[0],t.shape[1],new_n); z[:,:,:old_n]=t; expanded['model'][name]=z; expanded_names.append(name)
        elif name == 'deprel.scorer.W_bilin.bias':
            z=t.new_zeros(new_n); z[:old_n]=t; expanded['model'][name]=z; expanded_names.append(name)
        elif name.startswith('deprel_linear.') and t.ndim in (1,2) and t.shape[0]==old_n:
            z=t.new_zeros((new_n,)+tuple(t.shape[1:])); z[:old_n]=t; expanded['model'][name]=z; expanded_names.append(name)
    assert expanded_names, 'New labels exist but relation output tensor was not found'

args=copy.deepcopy(expanded['config'])
for key in list(args):
    if key.startswith('lora_') or key in ('peft_name','bert_lora'):
        args.pop(key,None)
args.update(use_peft=False, bert_finetune=True, optim='adamw', second_optim=None,
            lr=CFG['parser_lr'],
            # Stanza expresses the Transformer LR as a ratio to the parser LR.
            bert_learning_rate=CFG['transformer_lr']/CFG['parser_lr'],
            bert_start_finetuning=0, bert_warmup_steps=0,
            weight_decay=0.0, bert_weight_decay=0.0,
            max_grad_norm=CFG['max_grad_norm'], batch_size=CFG['batch_size'], seed=SEED,
            enable_gradient_checkpointing=True, augment_nopunct=0.0)
assert args['bert_model'] == HF_REPO

def one_dependency_path(model_type):
    deps=resources['zh-hans']['depparse']['gsdsimp_electra-large'].get('dependencies',[])
    names=[d['package'] for d in deps if d.get('model')==model_type]
    paths=[MODELS/'zh-hans'/model_type/f'{name}.pt' for name in names]
    paths=[p for p in paths if p.exists()]
    if len(paths)!=1: paths=list((MODELS/'zh-hans'/model_type).glob('*.pt'))
    assert len(paths)==1, f'Cannot unambiguously resolve {model_type}: {paths}'
    return paths[0]

if args.get('charlm'):
    args['charlm_forward_file']=str(one_dependency_path('forward_charlm').resolve())
    args['charlm_backward_file']=str(one_dependency_path('backward_charlm').resolve())
expanded.pop('bert_lora',None)
expanded['config']=args; expanded['global_step']=0; expanded['last_best_step']=0; expanded['dev_score_history']=[]
compat_path=OUT/'mandarin_electra_expanded_deprel_full_init.pt'
torch.save(expanded,compat_path,_use_new_zipfile_serialization=False)
del expanded; gc.collect(); torch.cuda.empty_cache()
print('original parser SHA-256:',sha256(base_path))
print('HF revision:',HF_COMMIT)
print('old relation labels:',old_n,'new labels:',new_labels,'expanded tensors:',expanded_names)


In [ ]:
# Load the initialization, audit it, and perform full Transformer + parser fine-tuning.
from stanza.models.depparse.trainer import GraphTrainer
from stanza.models.depparse.data import DataLoader, InfiniteBatch
from stanza.models.depparse.utils import predict_dataset
from stanza.models.common.pretrain import Pretrain
from stanza.utils.conll import CoNLL
from stanza.models.common.doc import HEAD, DEPREL
from stanza.models.depparse.transition.model import SubtreeCombination

pretrain_obj=Pretrain(filename=str(one_dependency_path('pretrain'))) if base_ckpt['config'].get('pretrain') else None
load_args=dict(args)
load_args.pop('transition_subtree_combination',None)
load_args['charlm_forward_file']=str(one_dependency_path('forward_charlm').resolve())
load_args['charlm_backward_file']=str(one_dependency_path('backward_charlm').resolve())
trainer=GraphTrainer.load(str(compat_path),pretrain=pretrain_obj,args=load_args,device=DEVICE,reset_history=True)
assert isinstance(trainer.args['transition_subtree_combination'],SubtreeCombination)

# Exact audit of every original parsing tensor and every old DEPREL slice.
loaded=trainer.model.get_params(skip_modules=True)
for name,old in base_ckpt['model'].items():
    got=loaded[name].detach().cpu()
    if name == 'deprel.scorer.W_bilin.weight': assert torch.equal(got[:,:,:old_n],old)
    elif name == 'deprel.scorer.W_bilin.bias' or (name.startswith('deprel_linear.') and old.ndim in (1,2) and old.shape[0]==old_n): assert torch.equal(got[:old_n],old)
    else: assert got.shape==old.shape and torch.equal(got,old),name

transformer=[(n,p) for n,p in trainer.model.named_parameters() if n.startswith('bert_model.')]
parser=[(n,p) for n,p in trainer.model.named_parameters() if not n.startswith('bert_model.')]
assert transformer and parser
assert all(p.requires_grad for _,p in transformer),[n for n,p in transformer if not p.requires_grad][:10]
assert any(p.requires_grad for _,p in parser)
assert not any('lora_' in n for n,_ in transformer)
trainable=sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total=sum(p.numel() for p in trainer.model.parameters())
print('Original Mandarin parser tensors preserved: OK')
print('trainable Transformer tensors:',len(transformer),'trainable total parameters:',trainable,'/',total)

train_doc=CoNLL.conll2doc(input_file=str(DATA/'train.predposlemma.conllu'))
dev_doc=CoNLL.conll2doc(input_file=str(DATA/'dev.predposlemma.conllu'))
train_loader=DataLoader(train_doc,CFG['batch_size'],trainer.args,pretrain_obj,vocab=trainer.vocab,evaluation=False,bert_tokenizer=trainer.model.bert_tokenizer)
dev_loader=DataLoader(dev_doc,CFG['batch_size'],trainer.args,pretrain_obj,vocab=trainer.vocab,evaluation=True,sort_during_eval=True,bert_tokenizer=trainer.model.bert_tokenizer)
infinite=InfiniteBatch(train_loader)

def predict_to_file(tr,loader,path):
    tr.model.eval()
    with torch.inference_mode(): preds=predict_dataset(tr,loader)
    loader.doc.set([HEAD,DEPREL],[y for x in preds for y in x])
    path.write_text(f'{loader.doc:C}\n\n',encoding='utf-8')
    return path

best_path=OUT/'best_dev_electra_yue_full_finetune.pt'
history=[]
dev_pred=OUT/'dev.full.step0000.conllu'
predict_to_file(trainer,dev_loader,dev_pred)
best_score=conll18(DATA/'dev.conllu',dev_pred)['LAS'].f1
best_step=0; trainer.save(str(best_path)); print('step 0 dev LAS',best_score*100)

for step in range(1,CFG['max_steps']+1):
    loss,_=trainer.update(infinite.next_batch(),eval=False)
    trainer.global_step=step
    if step % 20 == 0: print(f'step {step} loss {loss:.5f}')
    if step % CFG['eval_interval'] == 0:
        pred=OUT/f'dev.full.step{step:04d}.conllu'; predict_to_file(trainer,dev_loader,pred)
        score=conll18(DATA/'dev.conllu',pred)['LAS'].f1
        history.append({'step':step,'dev_conll18_las':score,'loss':float(loss),'prediction_sha256':sha256(pred)})
        print(f'== step {step}: dev CoNLL18 LAS {score*100:.2f}% ==')
        if score > best_score:
            best_score=score; best_step=step; trainer.save(str(best_path))
        if step-best_step >= CFG['patience_steps']:
            print('early stop'); break

(OUT/'full_dev_history.json').write_text(json.dumps(history,indent=2),encoding='utf-8')
print('BEST FULL DEV:',best_step,best_score*100,sha256(best_path))


## Dev-only comparison and manual review

When prompted, upload the exact `yue_dev_diagnostics.zip` produced previously. The comparison uses its verified LoRA dev prediction; it does not retrain LoRA and does not inspect test.

In the exported manual-review CSV, fill `manual_primary_diagnosis` / `manual_error_class` with categories such as `local attachment`, `clause scope`, `repair/disfluency`, `coordination`, `relation only`, or `annotation question`. The notebook deliberately does not infer these linguistic diagnoses from dependency distance alone.

In [ ]:
# Upload the prior LoRA DEV diagnostics, run the selected full model, and compare them.
# No test data is evaluated in this notebook.
import zipfile, unicodedata
import pandas as pd
from google.colab import files

print('Upload the existing yue_dev_diagnostics.zip')
uploaded=files.upload()
zip_names=[name for name in uploaded if name.endswith('.zip')]
assert len(zip_names)==1,zip_names
lora_zip=Path('/content')/zip_names[0]
EXPECTED_LORA_ZIP_SHA='6165dd6dbcb22d7871fc9f9a725eebf8c2a65cda95a20b9ae7cf449c9d882cc0'
EXPECTED_LORA_DEV_PRED_SHA='02e222057654b97dd3ced80d8c456014b84192185900c14ded78af7ceda420fd'
assert sha256(lora_zip)==EXPECTED_LORA_ZIP_SHA,(sha256(lora_zip),EXPECTED_LORA_ZIP_SHA)
with zipfile.ZipFile(lora_zip) as z:
    assert z.testzip() is None
    member='dev.predposlemma.best.pred.conllu'
    assert member in z.namelist()
    z.extract(member,WORK/'prior_lora')
lora_pred=WORK/'prior_lora'/member
assert sha256(lora_pred)==EXPECTED_LORA_DEV_PRED_SHA

del trainer; gc.collect(); torch.cuda.empty_cache()
best_trainer=GraphTrainer.load(str(best_path),pretrain=pretrain_obj,args=load_args,device=DEVICE)
full_doc=CoNLL.conll2doc(input_file=str(DATA/'dev.predposlemma.conllu'))
full_loader=DataLoader(full_doc,CFG['batch_size'],best_trainer.args,pretrain_obj,vocab=best_trainer.vocab,
                       evaluation=True,sort_during_eval=True,bert_tokenizer=best_trainer.model.bert_tokenizer)
full_pred=OUT/'dev.full.best.pred.conllu'
predict_to_file(best_trainer,full_loader,full_pred)
full_score=conll18(DATA/'dev.conllu',full_pred)['LAS'].f1
assert abs(full_score-best_score)<1e-12,(full_score,best_score)

def sentence_meta(block):
    out={}
    for line in block.splitlines():
        if line.startswith('# ') and ' = ' in line:
            k,v=line[2:].split(' = ',1); out[k]=v
    return out

gold_blocks=split_blocks((DATA/'dev.conllu').read_text(encoding='utf-8'))
input_blocks=split_blocks((DATA/'dev.predposlemma.conllu').read_text(encoding='utf-8'))
lora_blocks=split_blocks(lora_pred.read_text(encoding='utf-8'))
full_blocks=split_blocks(full_pred.read_text(encoding='utf-8'))
assert len(gold_blocks)==len(input_blocks)==len(lora_blocks)==len(full_blocks)==101
rows=[]
base=lambda x:x.split(':',1)[0]
for si,(gb,ib,lb,fb) in enumerate(zip(gold_blocks,input_blocks,lora_blocks,full_blocks),1):
    gr,ir,lr,fr=map(integer_rows,(gb,ib,lb,fb)); meta=sentence_meta(gb)
    assert len(gr)==len(ir)==len(lr)==len(fr)
    forms={int(x[0]):x[1] for x in gr}; forms[0]='ROOT'
    for g,inp,l,f in zip(gr,ir,lr,fr):
        assert g[:2]==inp[:2]==l[:2]==f[:2]
        tid=int(g[0]); gh,lh,fh=map(int,(g[6],l[6],f[6])); gd,ld,fd=g[7],l[7],f[7]
        dist=0 if gh==0 else abs(tid-gh)
        bucket='ROOT' if gh==0 else ('1-2' if dist<=2 else ('3-5' if dist<=5 else '6+'))
        l_uas=gh==lh; f_uas=gh==fh
        l_las=l_uas and base(gd)==base(ld); f_las=f_uas and base(gd)==base(fd)
        change='fixed' if (not l_las and f_las) else ('regressed' if (l_las and not f_las) else ('stable_correct' if l_las else 'persistent_error'))
        rows.append(dict(sentence_index=si,sent_id=meta.get('sent_id'),text=meta.get('text'),token_id=tid,
            form=g[1],predicted_upos=inp[3],gold_head=gh,gold_head_form=forms[gh],gold_deprel=gd,
            lora_head=lh,lora_head_form=forms.get(lh,'?'),lora_deprel=ld,lora_uas=l_uas,lora_las=l_las,
            full_head=fh,full_head_form=forms.get(fh,'?'),full_deprel=fd,full_uas=f_uas,full_las=f_las,
            distance=dist,distance_bucket=bucket,change=change,
            manual_error_class='',manual_notes=''))
df=pd.DataFrame(rows)

def aggregate(column):
    out=(df.groupby(column,dropna=False).agg(tokens=('token_id','size'),
         lora_UAS=('lora_uas','mean'),full_UAS=('full_uas','mean'),
         lora_LAS=('lora_las','mean'),full_LAS=('full_las','mean')).reset_index())
    out['UAS_delta_points']=100*(out.full_UAS-out.lora_UAS)
    out['LAS_delta_points']=100*(out.full_LAS-out.lora_LAS)
    return out

comparison=WORK/'comparison'; comparison.mkdir(exist_ok=True)
distance=aggregate('distance_bucket')
distance['distance_bucket']=pd.Categorical(distance.distance_bucket,['ROOT','1-2','3-5','6+'],ordered=True)
distance=distance.sort_values('distance_bucket')
relation=aggregate('gold_deprel').sort_values(['tokens','gold_deprel'],ascending=[False,True])
distance.to_csv(comparison/'distance_comparison.csv',index=False)
relation.to_csv(comparison/'relation_comparison.csv',index=False)
df.to_csv(comparison/'token_level_comparison.csv',index=False)

# Deterministically select 36 sentences.  Greedy coverage spans short/medium/long/root,
# fixed/regressed/persistent errors, and the predefined low-performing relations.
priority_rel={'advcl','reparandum','obl','conj','compound','parataxis','mark','vocative','amod','xcomp','mark:rel','obl:tmod'}
error_df=df[~(df.lora_las & df.full_las)].copy()
features={}
for si,g in error_df.groupby('sentence_index'):
    fs={'distance:'+x for x in g.distance_bucket.unique()}
    fs|={'change:'+x for x in g.change.unique() if x!='stable_correct'}
    fs|={'relation:'+x for x in g.gold_deprel.unique() if x in priority_rel}
    features[int(si)]=fs
universe=set().union(*features.values()) if features else set()
selected=[]; covered=set()
while len(selected)<CFG['selected_sentence_count'] and len(selected)<len(features):
    candidates=[si for si in features if si not in selected]
    best=max(candidates,key=lambda si:(len(features[si]-covered),len(error_df[error_df.sentence_index==si]),-si))
    selected.append(best); covered|=features[best]

review=df[df.sentence_index.isin(selected)].copy()
review['selection_order']=review.sentence_index.map({si:i+1 for i,si in enumerate(selected)})
review=review.sort_values(['selection_order','token_id'])
review.to_csv(comparison/'manual_review_36_sentences_token_rows.csv',index=False)
sentence_review=(review.groupby(['selection_order','sentence_index','sent_id','text'],dropna=False)
  .agg(tokens=('token_id','size'),lora_errors=('lora_las',lambda x:(~x).sum()),
       full_errors=('full_las',lambda x:(~x).sum()),fixed=('change',lambda x:(x=='fixed').sum()),
       regressed=('change',lambda x:(x=='regressed').sum()),max_gold_distance=('distance','max')).reset_index())
sentence_review['manual_primary_diagnosis']=''
sentence_review['manual_notes']=''
sentence_review.to_csv(comparison/'manual_review_36_sentences_index.csv',index=False)
(comparison/'manual_review_36_sentences.conllu').write_text(
    '\n\n'.join(gold_blocks[si-1] for si in selected)+'\n\n',encoding='utf-8')

lora_las=float(df.lora_las.mean()); full_las=float(df.full_las.mean())
fixed=int((df.change=='fixed').sum()); regressed=int((df.change=='regressed').sum())
eligible_rel=relation[relation.tokens>=10]
summary={'split':'dev','test_evaluated':False,'seed':SEED,
 'initialization':{'stanza_depparse_sha256':sha256(base_path),'hf_repo':HF_REPO,'hf_revision':HF_COMMIT,
                   'old_parser_weights_preserved':True,'new_deprel_labels':new_labels},
 'controls':{'same_frozen_predicted_pos_lemma_cache':True,'cache_sha256':cache_hashes,
             'same_train_dev_split':True,'same_parser_lr':CFG['parser_lr'],
             'same_transformer_lr':CFG['transformer_lr'],'same_dev_selection_rule':True},
 'lora':{'best_step':1200,'dev_conll18_las_percent':100*lora_las,'prediction_sha256':sha256(lora_pred)},
 'full_finetune':{'best_step':best_step,'dev_conll18_las_percent':100*full_las,
                  'prediction_sha256':sha256(full_pred),'checkpoint_sha256':sha256(best_path)},
 'full_minus_lora_las_points':100*(full_las-lora_las),'tokens_fixed':fixed,'tokens_regressed':regressed,
 'head_errors_fixed':int(((~df.lora_uas)&df.full_uas).sum()),
 'head_errors_regressed':int((df.lora_uas&(~df.full_uas)).sum()),
 'distance_buckets_las_improved':int((distance.LAS_delta_points>0).sum()),
 'distance_buckets_total':len(distance),
 'relations_support_ge_10_las_improved':int((eligible_rel.LAS_delta_points>0).sum()),
 'relations_support_ge_10_total':len(eligible_rel),
 'manual_review_sentences':len(selected),
 'interpretation_warning':'One matched schedule and one seed do not prove a general LoRA capacity limitation; full fine-tuning may have a different optimal learning rate.'}
(comparison/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))
display(distance,relation.head(25),sentence_review)


In [ ]:
# Package all reports and the full model.  The complete archive may be over 1 GB.
run_record={'config':CFG,'seed':SEED,'software':{'stanza':'1.14.0','transformers':'4.56.2','huggingface_hub':'0.34.4'},
            'input_hashes':{'train':SPLIT_SHA256['train'],'dev':SPLIT_SHA256['dev'],
                            'train_predposlemma':cache_hashes['train'],'dev_predposlemma':cache_hashes['dev']},
            'hf_revision':HF_COMMIT,'test_evaluated':False}
(OUT/'run_record.json').write_text(json.dumps(run_record,ensure_ascii=False,indent=2),encoding='utf-8')

small=WORK/'analysis_bundle'; small.mkdir(exist_ok=True)
for p in comparison.iterdir(): shutil.copy2(p,small/p.name)
for p in (OUT/'full_dev_history.json',OUT/'run_record.json',full_pred): shutil.copy2(p,small/p.name)
small_zip=shutil.make_archive('/content/yue_full_vs_lora_dev_analysis','zip',root_dir=small)

complete=WORK/'complete_bundle'; complete.mkdir(exist_ok=True)
for p in small.iterdir(): shutil.copy2(p,complete/p.name)
shutil.copy2(best_path,complete/best_path.name)
complete_zip=shutil.make_archive('/content/yue_full_finetune_complete','zip',root_dir=complete)
print('analysis archive:',small_zip,sha256(small_zip))
print('complete archive:',complete_zip,sha256(complete_zip))
print('The complete archive includes the dev-selected full-fine-tuned model. Keep its printed SHA-256.')
files.download(small_zip)
files.download(complete_zip)
